# GGIS 407 - Lab 5: PostGIS
#### Interactive Jupyter Notebook

In this notebook we will use <b>PostGIS</b> to learn the basic concepts of spatial databases. Work through the PostgreSQL notebook first.

## Notebook Outline:
- [Introduction](#intro)
- [Install PostgreSQL](#run_once) (first time only)
- [Setup](#setup) (run this first!)
- [Database Basics](#postgreSQL)
    - [Create Database](#create_db)
    - [Connect to Database](#connect_db)
    - [Initialize PostGIS](#init_postgis)
    - [View Databases and Tables](#view_db)
- [Database Operations](#op_db)
    - [Create a PostGIS Table and Add Column](#create_table)
    - [Review Metadata](#meta)
    - [Insert Data into Table](#insert_data)
    - [Query Table](#list_data)
- [PostGIS Example](#example)
    - [Import Data from a Shapefile](#import_data)
    - [Query Data](#qu_data)
    - [Spatial Operations](#sp_op)

<a id='intro'></a>
## Introduction

<b>PostGIS</b> is a spatial database extender for PostgreSQL. It adds support for geographic objects, allowing location queries to be run in SQL.

Useful documentation:

>PostGIS https://postgis.net/
<br>PostGIS Documentation https://postgis.net/documentation/
<br>Introduction to PostGIS https://postgis.net/workshops/postgis-intro/

<a id='run_once'></a>
## Install PostgreSQL (first time only)
This lab runs its own small PostgreSQL database server inside your I-GUIDE JupyterHub account. The next cell installs what it needs:

- the PostgreSQL server and the PostGIS extension, into a folder called `pgenv` in your home directory (**this takes a few minutes the first time**; after that it is skipped), and
- two Python packages: `jupysql`, which provides the `%sql` commands used in this notebook, and `psycopg2`, which lets Python talk to PostgreSQL.

If the cell says it installed or removed Python packages, choose <b>Kernel -> Restart Kernel</b>, then continue with Setup.

In [ ]:
import os, shutil, subprocess
from importlib.metadata import version, PackageNotFoundError

def installed(pkg):
    try:
        version(pkg)
        return True
    except PackageNotFoundError:
        return False

changed = False

# 1. Python packages for the %sql commands
if installed('ipython-sql'):   # an older %sql package that conflicts with jupysql
    get_ipython().run_line_magic('pip', 'uninstall -y -q ipython-sql')
    changed = True
needed = []
if not installed('jupysql'):
    needed.append('jupysql')
if not (installed('psycopg2') or installed('psycopg2-binary')):
    needed.append('psycopg2-binary')
if needed:
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(needed))
    changed = True

# 2. PostgreSQL server and PostGIS, installed once into ~/pgenv
PGENV = os.path.expanduser('~/pgenv')
if not os.path.exists(os.path.join(PGENV, 'bin', 'pg_ctl')):
    installer = shutil.which('mamba') or shutil.which('conda')
    print('Installing PostgreSQL and PostGIS into ~/pgenv. This takes a few minutes the first time...')
    subprocess.run([installer, 'create', '-y', '-q', '-p', PGENV, '-c', 'conda-forge',
                    'postgresql=16', 'postgis'], check=True)

if changed:
    print('Python packages were installed or removed: choose Kernel -> Restart Kernel, then run Setup.')
else:
    print('Everything is installed. Continue with Setup.')

<a id='setup'></a>
## Setup
Run this cell every time you open the notebook. It creates the database cluster the first time (in a folder called `pgdata` in your home directory) and starts the PostgreSQL server if it isn't already running. Both Lab 5 notebooks use the same server.

In [ ]:
# set up and start the PostgreSQL server
import os, subprocess

PGENV = os.path.expanduser('~/pgenv')
os.environ['PATH'] = os.path.join(PGENV, 'bin') + os.pathsep + os.environ['PATH']
os.environ['PGDATA'] = os.path.expanduser('~/pgdata')   # where PostgreSQL keeps its files
os.environ['PGHOST'] = 'localhost'                      # the server runs on this machine
os.environ['PGPORT'] = '5432'                           # PostgreSQL's standard port
pgdata = os.environ['PGDATA']

# create the database cluster (only the first time)
if not os.path.exists(os.path.join(pgdata, 'PG_VERSION')):
    !initdb --auth=trust --encoding=UTF8 --no-locale > /dev/null && echo "Database cluster created in ~/pgdata"

# start the server if it isn't running yet
if subprocess.run(['pg_isready', '-q']).returncode == 0:
    print('PostgreSQL server is already running.')
else:
    # a lock file left over from an earlier session can stop the server from starting
    pid_file = os.path.join(pgdata, 'postmaster.pid')
    if os.path.exists(pid_file):
        os.remove(pid_file)
    !pg_ctl --log={pgdata}/server.log --wait start

# load the %sql commands and show up to 20 rows of each result
%reload_ext sql
%config SqlMagic.displaylimit = 20

<a id='postgreSQL'></a>
## Database Basics

<a id='create_db'></a>
### Create Database

In [ ]:
# Create a new database called postgis_lab5
# (if you run this again, it reports that the database already exists; that's fine)
!createdb postgis_lab5

<a id='connect_db'></a>
### Connect to Database

In [ ]:
# Establish a connection to the database you have created
%sql postgresql+psycopg2://localhost:5432/postgis_lab5

<a id='init_postgis'></a>
### Initialize PostGIS

PostGIS is an <b>extension</b> to PostgreSQL: it is installed with the server, but each database has to switch it on before it can use spatial data types and functions.

In [ ]:
%%sql
create extension if not exists postgis;
select postgis_full_version();

<a id='view_db'></a>
### View Databases and Tables

In [ ]:
# View all the databases on the server
!psql -d postgis_lab5 -c "\l"

In [ ]:
# List all the tables and views in the database.
# PostGIS adds a table of coordinate reference systems (spatial_ref_sys)
# and views that describe spatial columns (geometry_columns, geography_columns).
!psql -d postgis_lab5 -c "\d"

In [ ]:
# Look up one coordinate reference system: EPSG:3174, NAD83 / Great Lakes Albers
%sql select srid, auth_name, auth_srid, srtext from spatial_ref_sys where srid=3174

<a id='op_db'></a>
## Database Operations

<a id='create_table'></a>
### Create a PostGIS Table and Add Column

Create a new table called mypostgis in the postgis_lab5 database, then add a second geometry column to it.

In [ ]:
%%sql
-- Create a new table called mypostgis, deleting any old table with the same name first
drop table if exists mypostgis;
create table mypostgis (id int4, geometry geometry(POLYGON, 4326));

In [ ]:
# Add a column called centroid that holds 2-D points in EPSG:4326
%sql SELECT AddGeometryColumn('mypostgis', 'centroid', 4326, 'POINT', 2);

In [ ]:
# Check that the table was created
!psql -d postgis_lab5 -c "\dt"

<a id="meta"></a>
### Review Metadata

Review the metadata (information schema) of your table. First we view details of all the columns, and then view the information specifically about spatial columns.

In [ ]:
%sql select column_name, data_type, udt_name from information_schema.columns where table_name = 'mypostgis';

In [ ]:
%sql select * from geometry_columns where f_table_name='mypostgis';

<a id='insert_data'></a>
### Insert Data into Table

Load a few rows into the table. `ST_GeomFromText` builds a geometry from Well-Known Text (WKT) and a coordinate reference system code.

In [ ]:
%%sql
-- Insert a row of data into the table
insert into mypostgis (id, geometry, centroid) values (1,
    ST_GeomFromText('POLYGON((-128 50, -128 49.948, -127.874 49.948, -127.874 50, -128 50))', 4326),
    ST_GeomFromText('POINT(-127.937 49.974)', 4326));

In [ ]:
# View the data in the table
%sql select * from mypostgis

In [ ]:
%%sql
-- Insert another row
insert into mypostgis (id, geometry, centroid) values (2,
    ST_GeomFromText('POLYGON((-130 50, -130 20, -70 20, -70 50, -130 50))', 4326),
    ST_GeomFromText('POINT(-100 35)', 4326));

<a id='list_data'></a>
### Query Table

In [ ]:
# Review the contents of the table: geometries are stored in a compact binary format
%sql select * from mypostgis;

In [ ]:
# View the data in the table in a human-readable format
%sql select id, ST_AsText(geometry), ST_AsText(centroid) from mypostgis;

In [ ]:
# Check our centroids: does the stored point match the polygon's real centroid?
%sql select id, ST_AsText(centroid) as stored, ST_AsText(ST_Centroid(geometry)) as calculated from mypostgis;

<a id="example"></a>
## PostGIS Example

<a id="import_data"></a>
### Import Data from a Shapefile

We will use the boundaries of the 1,296 cities, villages and towns in Illinois (U.S. Census Bureau, 2022), which are in the `data` folder next to this notebook. `shp2pgsql` is a tool that comes with PostGIS: it turns a shapefile into SQL commands, and the `|` sends those commands to `psql` to run in our database.

- `-s 4269` tells PostGIS the data's coordinate reference system (NAD83, EPSG:4269)
- `-I` creates a spatial index on the geometry column

The first line removes any `towns` table left from an earlier run, so you can run this cell again.

In [ ]:
# Import a shapefile into PostGIS as a table called towns
!psql -q -d postgis_lab5 -c "DROP TABLE IF EXISTS towns"
!shp2pgsql -s 4269 -I data/IL_Municipalities_2022.shp public.towns | psql -q -d postgis_lab5

In [ ]:
# View the tables: you will see the new towns table
!psql -d postgis_lab5 -c "\dt"

<a id="qu_data"></a>
### Query Data

`shp2pgsql` turns the shapefile's column names into lower case and stores the geometry in a column called `geom`.

In [ ]:
# View a few sample rows in the new table
%sql select gid, geoid, name, namelsad, type, aland, awater from towns limit 5

In [ ]:
# Find the number of towns in the towns table
%sql SELECT count(*) FROM towns;

In [ ]:
# Count the towns of each type
%sql SELECT type, count(*) FROM towns GROUP BY type ORDER BY count(*) DESC;

In [ ]:
# Display the names of all the towns, in alphabetical order
%sql SELECT name FROM towns ORDER BY name;

Notice that `count(name)` counts only the rows where `name` is not NULL, while `count(*)` counts every row. In this dataset every town has a name, so they are equal; in messier data they can differ, and you might remove the rows with no name:

In [ ]:
%sql SELECT count(name) AS with_name, count(*) AS all_rows FROM towns;

In [ ]:
# Delete towns with no name (none here, so 0 rows are affected)
%sql delete from towns where name is NULL

In [ ]:
# Find all information about the city of Urbana
%sql SELECT gid, geoid, name, namelsad, type, aland, awater FROM towns WHERE name = 'Urbana';

In [ ]:
# Display the geometry of Urbana in a human-readable format (the first 300 characters)
%sql SELECT left(ST_AsText(geom), 300) FROM towns WHERE name = 'Urbana';

In [ ]:
# Display the geometry of Urbana in GeoJSON format (the first 300 characters)
%sql SELECT left(ST_AsGeoJSON(geom), 300) FROM towns WHERE name = 'Urbana';

<a id="sp_op"></a>
### Spatial Operations

In [ ]:
# Find the area of Urbana
%sql SELECT ST_Area(geom) FROM towns WHERE name = 'Urbana';

<b>What are the units? Let's find out.</b> First we find the coordinate reference system of the data.

In [ ]:
%sql select * from geometry_columns where f_table_name='towns';

Next, we find the units by looking at the details of that reference system.

In [ ]:
%sql select srid, srtext from spatial_ref_sys where srid=4269;

So the units are <b>degrees</b>, and the area above is in square degrees, which isn't a useful unit.

How do we calculate the area of Urbana in square kilometers? First we transform the data into a projected coordinate system that uses meters (EPSG:3174, Great Lakes Albers, an equal-area projection), then we calculate the area and divide by 1,000,000.

In [ ]:
%sql select ST_Area(ST_Transform(geom, 3174))/1000000 as area_sq_km from towns where name = 'Urbana';

Compare this with the land area in the `aland` column (in square meters), which the Census Bureau calculated: the numbers should be close. Now let us find the combined area of all the towns.

In [ ]:
%sql SELECT ST_Area(ST_Union(ST_Transform(geom, 3174)))/1000000 as total_area_sq_km FROM towns;

Find the <b>bounding box</b> that encloses Urbana, and the bounding box that encloses all the towns.

In [ ]:
# Bounding box of Urbana
%sql select ST_Extent(geom) from towns where name = 'Urbana';

In [ ]:
# Bounding box of all the towns in Illinois
%sql select ST_Extent(geom) from towns;

Next, we will see how to <b>project</b> data. Let us project the geometry of Urbana to Great Lakes Albers (EPSG:3174) and show it in a human-readable format. Notice that the coordinates are now in meters.

In [ ]:
%sql select left(ST_AsText(ST_Transform(geom, 3174)), 300) from towns where name = 'Urbana';

Let us see how to calculate <b>centroids</b>. Here are the centroids of 10 towns in a human-readable format:

In [ ]:
%sql select name, ST_AsText(ST_Centroid(geom)) from towns limit 10;

Next, we will do a spatial operation: <b>list the names and centroids of all the towns within 15 km of the centroid of Urbana</b>. We will need multiple steps for this.

1. Find the centroid of Urbana.

In [ ]:
%sql select name, ST_AsText(ST_Centroid(geom)) from towns where name = 'Urbana';

In [ ]:
%sql select name, ST_X(ST_Centroid(geom)), ST_Y(ST_Centroid(geom)) from towns where name = 'Urbana';

In [ ]:
%%sql
-- the same centroid in Great Lakes Albers (meters)
select name, ST_X(ST_Centroid(ST_Transform(geom, 3174))), ST_Y(ST_Centroid(ST_Transform(geom, 3174)))
from towns where name = 'Urbana';

2. Next, find the towns that are within 15 km of that centroid. `ST_DWithin(a, b, distance)` is true when two geometries are within the given distance of each other.

Note that we need to reproject into a system that has units in meters, so we can give the distance as 15,000 meters. The coordinates in the queries below are the ones from step 1.

In [ ]:
%%sql
-- using the centroid in latitude/longitude (EPSG:4269), transformed to meters
select name, ST_X(ST_Centroid(geom)), ST_Y(ST_Centroid(geom)) from towns
where ST_DWithin(ST_Transform(geom, 3174),
                 ST_Transform(ST_GeomFromText('POINT(-88.197280776335 40.11068293832863)', 4269), 3174),
                 15000);

In [ ]:
%%sql
-- the same query using the centroid already in Great Lakes Albers (EPSG:3174)
select name, ST_X(ST_Centroid(geom)), ST_Y(ST_Centroid(geom)) from towns
where ST_DWithin(ST_Transform(geom, 3174),
                 ST_GeomFromText('POINT(680351.746457213 400888.8269733935)', 3174),
                 15000);

3. Instead of copying coordinates by hand, a <b>subquery</b> can calculate Urbana's centroid inside the same query:

In [ ]:
%%sql
select name, round(ST_Distance(ST_Transform(geom, 3174), u.center)::numeric / 1000, 1) as distance_km
from towns,
     (select ST_Centroid(ST_Transform(geom, 3174)) as center from towns where name = 'Urbana') as u
where ST_DWithin(ST_Transform(geom, 3174), u.center, 15000)
order by distance_km;

## Try it
1. Which five towns in Illinois have the largest area in square kilometers? (Hint: `order by` and `limit`.)
2. How many towns are within 25 km of the centroid of Champaign?
3. Look up the EPSG code for "NAD83 / Illinois East" and repeat the area calculation for Urbana in that system. How close is it to the Great Lakes Albers result?

In [ ]:
# Practice what you have learned about PostGIS
# Feel free to modify any code above

## Stopping the Server (optional)
The server stops on its own when your I-GUIDE JupyterHub session ends. To stop it yourself, run:

In [ ]:
!pg_ctl stop